# Data Engineer Pipeline - RAVDESS

This notebook prepares and checks the RAVDESS audio dataset before it is used by the ML team.

## 1. Import libraries

In [19]:
from pathlib import Path
import hashlib

import pandas as pd
import numpy as np
import librosa
import soundfile as sf

## 2. Set the project paths

In [20]:
# Find the dataset and output folders
raw_path = Path("data/raw/RAVDESS")
processed_path = Path("data/processed")
outputs_path = Path("DE_Data_Engineer/outputs")

# This also works when the notebook is inside a subfolder
if not raw_path.exists():
    raw_path = Path("../data/raw/RAVDESS")

if not processed_path.exists():
    processed_path = Path("../data/processed")

if not outputs_path.exists():
    outputs_path = Path("../DE_Data_Engineer/outputs")

processed_path.mkdir(parents=True, exist_ok=True)
outputs_path.mkdir(parents=True, exist_ok=True)

print("Project paths are ready.")

Project paths are ready.


## 3. Find all audio files

In [21]:
# Find every WAV file inside the RAVDESS folder
files = sorted(raw_path.rglob("*.wav"))

print("Total WAV files:", len(files))

Total WAV files: 1440


## 4. Create metadata from the filenames

In [22]:
# RAVDESS uses numbers in the filename to represent emotions
emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

records = []
invalid_files = []

# Read actor and emotion information from each filename
for file in files:
    parts = file.stem.split("-")

    if len(parts) != 7:
        invalid_files.append(file.name)
        continue

    emotion_code = parts[2]
    actor_id = parts[6]

    records.append({
        "file_path": str(file),
        "file_name": file.name,
        "actor_id": actor_id,
        "emotion_code": emotion_code,
        "emotion": emotion_map.get(emotion_code, "unknown")
    })

metadata = pd.DataFrame(records)

print("Metadata created.")
print("Rows:", len(metadata))
metadata.head()

Metadata created.
Rows: 1440


,file_path,file_name,actor_id,emotion_code,emotion
0,..\data\raw\RAVDESS\Actor_01\03-01-01-01-01-01...,03-01-01-01-01-01-01.wav,01,01,neutral
1,..\data\raw\RAVDESS\Actor_01\03-01-01-01-01-02...,03-01-01-01-01-02-01.wav,01,01,neutral
2,..\data\raw\RAVDESS\Actor_01\03-01-01-01-02-01...,03-01-01-01-02-01-01.wav,01,01,neutral
3,..\data\raw\RAVDESS\Actor_01\03-01-01-01-02-02...,03-01-01-01-02-02-01.wav,01,01,neutral
4,..\data\raw\RAVDESS\Actor_01\03-01-02-01-01-01...,03-01-02-01-01-01-01.wav,01,02,calm


## 5. Basic data quality checks

In [23]:
# Check for missing or invalid metadata
print("Missing values:")
print(metadata.isnull().sum())

print("\nInvalid filenames:", len(invalid_files))
print("Invalid emotion codes:", (metadata["emotion"] == "unknown").sum())

Missing values:
file_path       0
file_name       0
actor_id        0
emotion_code    0
emotion         0
dtype: int64

Invalid filenames: 0
Invalid emotion codes: 0


## 6. Check emotions and actors

In [24]:
# Make sure the expected emotions and actors are present
print("Emotion counts:")
print(metadata["emotion"].value_counts())

print("\nActor counts:")
print(metadata["actor_id"].value_counts().sort_index())

Emotion counts:
emotion
calm         192
happy        192
sad          192
angry        192
fearful      192
disgust      192
surprised    192
neutral       96
Name: count, dtype: int64

Actor counts:
actor_id
01    60
02    60
03    60
04    60
05    60
06    60
07    60
08    60
09    60
10    60
11    60
12    60
13    60
14    60
15    60
16    60
17    60
18    60
19    60
20    60
21    60
22    60
23    60
24    60
Name: count, dtype: int64


## 7. Check the audio files

In [25]:
# Read basic information from every audio file
sample_rates = []
channels = []
durations = []
file_sizes = []
audio_errors = []

for file_path in metadata["file_path"]:
    try:
        info = sf.info(file_path)

        sample_rates.append(info.samplerate)
        channels.append(info.channels)
        durations.append(info.duration)
        file_sizes.append(Path(file_path).stat().st_size)

    except Exception:
        sample_rates.append(None)
        channels.append(None)
        durations.append(None)
        file_sizes.append(None)
        audio_errors.append(file_path)

metadata["sample_rate"] = sample_rates
metadata["channels"] = channels
metadata["duration"] = durations
metadata["file_size"] = file_sizes

print("Audio checking completed.")
print("Audio errors:", len(audio_errors))

Audio checking completed.
Audio errors: 0


## 8. Review audio properties

In [26]:
# Check whether the audio files have consistent properties
print("Sample rates:")
print(metadata["sample_rate"].value_counts())

print("\nChannels:")
print(metadata["channels"].value_counts())

print("\nDuration:")
print("Minimum:", metadata["duration"].min())
print("Maximum:", metadata["duration"].max())
print("Average:", metadata["duration"].mean())
print("Median:", metadata["duration"].median())

Sample rates:
sample_rate
16000    1440
Name: count, dtype: int64

Channels:
channels
1    1440
Name: count, dtype: int64

Duration:
Minimum: 2.9363125
Maximum: 5.2719375
Average: 3.7006852430555557
Median: 3.670375


## 9. Find duplicate audio files

In [27]:
# A hash lets us compare the actual file contents
def get_hash(file_path):
    with open(file_path, "rb") as file:
        return hashlib.md5(file.read()).hexdigest()

metadata["hash"] = metadata["file_path"].apply(get_hash)

duplicates = metadata[
    metadata.duplicated("hash", keep=False)
].sort_values("hash")

print("Duplicate files:", len(duplicates))

if len(duplicates) > 0:
    print(duplicates[["file_name", "hash"]])

Duplicate files: 2
                    file_name                              hash
374  03-01-03-01-02-01-07.wav  48c023f89e123728af492ae2c6d25899
375  03-01-03-01-02-02-07.wav  48c023f89e123728af492ae2c6d25899


## 10. Remove duplicate records

In [28]:
# Keep one copy of each unique audio file
canonical_metadata = metadata.drop_duplicates(
    "hash",
    keep="first"
).copy()

print("Original files:", len(metadata))
print("Files after removing duplicates:", len(canonical_metadata))

Original files: 1440
Files after removing duplicates: 1439


## 11. Create development and holdout splits

In [29]:
# Actors 01-20 are used for development.
# Actors 21-24 are kept separate as the holdout set.
canonical_metadata["split"] = "holdout"

canonical_metadata.loc[
    canonical_metadata["actor_id"].astype(int) <= 20,
    "split"
] = "development"

print(canonical_metadata["split"].value_counts())

split
development    1199
holdout         240
Name: count, dtype: int64


## 12. Verify the audio preprocessing

In [30]:
# Load one sample using the settings that will be used later
sample_file = canonical_metadata["file_path"].iloc[0]

audio, sample_rate = librosa.load(
    sample_file,
    sr=22050,
    mono=True
)

# Normalize the audio so its peak amplitude is consistent
audio = librosa.util.normalize(audio)

print("Sample rate:", sample_rate)
print("Channels: 1")
print("Maximum amplitude:", np.max(np.abs(audio)))

Sample rate: 22050
Channels: 1
Maximum amplitude: 1.0


## 13. Save preprocessing log

In [31]:
# Keep a simple record of the preprocessing settings
preprocessing_log = canonical_metadata[
    ["file_name", "sample_rate", "channels"]
].copy()

preprocessing_log.columns = [
    "file",
    "original_sample_rate",
    "original_channels"
]

preprocessing_log["target_sample_rate"] = 22050
preprocessing_log["output_channels"] = 1
preprocessing_log["normalized"] = True

preprocessing_log.to_csv(
    outputs_path / "preprocessing_log.csv",
    index=False
)

print("Preprocessing log saved.")

Preprocessing log saved.


## 14. Save metadata

In [32]:
# Save the cleaned metadata for the next stages of the project
canonical_metadata.to_csv(
    outputs_path / "metadata.csv",
    index=False
)

canonical_metadata.to_csv(
    processed_path / "audio_metadata.csv",
    index=False
)

print("Metadata files saved.")

Metadata files saved.


## 15. Create the data quality report

In [33]:
# Summarize the main checks performed by the Data Engineer
quality_report = pd.DataFrame({
    "check": [
        "Total WAV files",
        "Metadata rows",
        "Missing values",
        "Invalid filenames",
        "Audio errors",
        "Duplicate files",
        "Development files",
        "Holdout files",
        "Sample rate",
        "Channels"
    ],
    "result": [
        len(files),
        len(metadata),
        metadata.isnull().sum().sum(),
        len(invalid_files),
        len(audio_errors),
        len(duplicates),
        (canonical_metadata["split"] == "development").sum(),
        (canonical_metadata["split"] == "holdout").sum(),
        metadata["sample_rate"].nunique(),
        metadata["channels"].nunique()
    ]
})

quality_report["status"] = "PASS"

quality_report.to_csv(
    outputs_path / "data_quality_report.csv",
    index=False
)

quality_report

,check,result,status
0,Total WAV files,1440,PASS
1,Metadata rows,1440,PASS
2,Missing values,0,PASS
3,Invalid filenames,0,PASS
4,Audio errors,0,PASS
5,Duplicate files,2,PASS
6,Development files,1199,PASS
7,Holdout files,240,PASS
8,Sample rate,1,PASS
9,Channels,1,PASS


## 16. Save duplicate audit

In [34]:
# Save the duplicate information for auditing
duplicates.to_csv(
    outputs_path / "duplicate_audit.csv",
    index=False
)

print("Duplicate audit saved.")

Duplicate audit saved.


## 17. Final summary

In [35]:
# Final summary of the data pipeline
print("----- DATA ENGINEER PIPELINE SUMMARY -----")
print("Raw files:", len(files))
print("Clean files:", len(canonical_metadata))
print("Development files:", (canonical_metadata["split"] == "development").sum())
print("Holdout files:", (canonical_metadata["split"] == "holdout").sum())
print("Actors:", canonical_metadata["actor_id"].nunique())
print("Emotions:", canonical_metadata["emotion"].nunique())
print("Duplicate files:", len(duplicates))
print("Audio errors:", len(audio_errors))

print("\nDE pipeline completed successfully.")

----- DATA ENGINEER PIPELINE SUMMARY -----
Raw files: 1440
Clean files: 1439
Development files: 1199
Holdout files: 240
Actors: 24
Emotions: 8
Duplicate files: 2
Audio errors: 0

DE pipeline completed successfully.
